# Exercise 06 — Performance & Best Practices

**numpy-mastery** · 10 problems · Easy → Hard

**Scope**: only concepts from `notebooks/06_performance_tricks.ipynb`  
→ views vs copies, anti-patterns, memory layout, stride tricks, file I/O,
→ vectorisation, `np.shares_memory`, `np.ascontiguousarray`

**Rules**:
- No loops unless the problem explicitly benchmarks them against a vectorised solution
- Use `time.perf_counter()` for timing

---


In [ ]:
import numpy as np
import time
import tempfile, os

def timer(fn, *args, repeat=5, **kwargs):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        times.append((time.perf_counter() - t0) * 1000)
    return min(times), result


---
### Problem 01 · Easy — identify view or copy
For each operation below, predict and then verify whether the result
is a **view** or a **copy** of `arr`. Fill in `True` (view) or `False` (copy).


In [ ]:
arr = np.arange(24).reshape(4, 6)

a = arr[1:3, :]        # slice
b = arr[[0, 2], :]     # fancy index
c = arr.T              # transpose
d = arr.reshape(6, 4)  # reshape
e = arr[arr > 10]      # boolean mask

# YOUR CODE — True if view, False if copy
is_view_a = None
is_view_b = None
is_view_c = None
is_view_d = None
is_view_e = None


In [ ]:
assert is_view_a == np.shares_memory(a, arr)
assert is_view_b == np.shares_memory(b, arr)
assert is_view_c == np.shares_memory(c, arr)
assert is_view_d == np.shares_memory(d, arr)
assert is_view_e == np.shares_memory(e, arr)
print(f"✓ Problem 01 passed")
print(f"  slice={is_view_a}, fancy={is_view_b}, T={is_view_c}, reshape={is_view_d}, bool={is_view_e}")


---
### Problem 02 · Easy — fix the mutation bug
The function below has a bug: it modifies the caller's array.
Fix it so `original` remains unchanged after the call.


In [ ]:
def normalise_buggy(arr):
    arr -= arr.mean()   # ← mutates the input!
    arr /= arr.std()
    return arr

original = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
backup   = original.copy()

# YOUR CODE — write normalise_fixed(arr) that does NOT mutate its input
def normalise_fixed(arr):
    pass   # replace with your implementation


In [ ]:
result = normalise_fixed(original)
assert np.array_equal(original, backup), "original was mutated!"
assert np.isclose(result.mean(), 0.0, atol=1e-10)
assert np.isclose(result.std(),  1.0, atol=1e-10)
print("✓ Problem 02 passed — original unchanged, result is z-scored")


---
### Problem 03 · Easy — fix the append anti-pattern
The function below builds an array by appending inside a loop.
Rewrite it as `fast_version` using pre-allocation or vectorisation.
Both must return the same result.


In [ ]:
def slow_version(n):
    result = np.array([])
    for i in range(n):
        result = np.append(result, i ** 2)
    return result

# YOUR CODE
def fast_version(n):
    pass   # no loops, no np.append

N = 10_000
slow = slow_version(N)
fast = fast_version(N)


In [ ]:
assert np.array_equal(slow, fast), "Results must match"
t_slow, _ = timer(slow_version, N)
t_fast, _ = timer(fast_version, N)
speedup = t_slow / t_fast
assert speedup > 5, f"Expected >5x speedup, got {speedup:.1f}x"
print(f"✓ Problem 03 passed — speedup: {speedup:.1f}x")


---
### Problem 04 · Easy — vectorise a loop
Replace the loop below with a single vectorised expression.
Both must return the same result.


In [ ]:
arr = np.random.default_rng(0).random(500_000)

def loop_version(a):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = 1.0 / (1.0 + np.exp(-a[i]))   # sigmoid
    return out

# YOUR CODE
def vec_version(a):
    pass   # one line, no loop


In [ ]:
expected = loop_version(arr)
result   = vec_version(arr)
assert np.allclose(result, expected)
t_loop, _ = timer(loop_version, arr)
t_vec,  _ = timer(vec_version,  arr)
speedup = t_loop / t_vec
assert speedup > 20, f"Expected >20x speedup, got {speedup:.1f}x"
print(f"✓ Problem 04 passed — speedup: {speedup:.1f}x")


---
### Problem 05 · Medium — contiguous layout
Given `arr` (C-contiguous) and `arr_T = arr.T` (no longer C-contiguous):
1. Verify `arr` is C-contiguous and `arr_T` is NOT
2. Create `arr_T_c` — a C-contiguous copy of `arr_T`
3. Show that summing `arr_T_c` along axis=0 is faster than summing `arr_T` along axis=0


In [ ]:
rng = np.random.default_rng(0)
arr   = rng.random((2000, 2000))
arr_T = arr.T

# YOUR CODE
arr_T_c = None   # C-contiguous version of arr_T

# Timing
t_noncontig, _ = timer(lambda: arr_T.sum(axis=0))
t_contig,    _ = timer(lambda: arr_T_c.sum(axis=0))


In [ ]:
assert arr.flags['C_CONTIGUOUS']  == True
assert arr_T.flags['C_CONTIGUOUS'] == False
assert arr_T_c.flags['C_CONTIGUOUS'] == True
assert np.allclose(arr_T.sum(axis=0), arr_T_c.sum(axis=0))
print(f"✓ Problem 05 passed")
print(f"  non-contiguous sum: {t_noncontig:.3f} ms")
print(f"  contiguous sum    : {t_contig:.3f} ms")


---
### Problem 06 · Medium — stride tricks: sliding window
Using `np.lib.stride_tricks.as_strided`, build a **zero-copy** view of
`arr` with shape `(N - W + 1, W)` where each row is one sliding window
of width `W`. Set `writeable=False`.

Then compute `window_max` — the maximum of each window — in one line.


In [ ]:
from numpy.lib.stride_tricks import as_strided

arr = np.array([3, 1, 4, 1, 5, 9, 2, 6, 5, 3], dtype=float)
W = 3

# YOUR CODE
windows    = None   # shape (8, 3), zero-copy view
window_max = None   # shape (8,)


In [ ]:
assert windows.shape == (len(arr) - W + 1, W)
assert np.shares_memory(windows, arr), "Must be a view, not a copy"
assert windows.flags['WRITEABLE'] == False
expected_max = np.array([4., 4., 5., 9., 9., 9., 6., 6.])
assert np.array_equal(window_max, expected_max), f"Got {window_max}"
print("✓ Problem 06 passed")
print("windows    :", windows)
print("window_max :", window_max)


---
### Problem 07 · Medium — stride tricks: 2-D patches
Using `as_strided`, create a view of image `img` (shape `(H, W)`) that
produces all `(kH, kW)` patches — shape `(H-kH+1, W-kW+1, kH, kW)`.
This is the core of how convolution is implemented without loops.


In [ ]:
from numpy.lib.stride_tricks import as_strided

img = np.arange(25, dtype=float).reshape(5, 5)
kH, kW = 3, 3

H, W = img.shape
oH, oW = H - kH + 1, W - kW + 1

# YOUR CODE
patches = None   # shape (3, 3, 3, 3)


In [ ]:
assert patches.shape == (oH, oW, kH, kW), f"Got {patches.shape}"
assert np.shares_memory(patches, img), "Must be a view"
# Verify: top-left patch should be img[0:3, 0:3]
assert np.array_equal(patches[0, 0], img[0:3, 0:3])
# Bottom-right patch should be img[2:5, 2:5]
assert np.array_equal(patches[2, 2], img[2:5, 2:5])
print("✓ Problem 07 passed")
print("patches[0,0]:\n", patches[0, 0])


---
### Problem 08 · Medium — file I/O round-trip
Save `X` and `y` to a compressed `.npz` file, reload them, and verify
the round-trip is exact. Also save `X` alone to a `.npy` file and verify.
Report the file sizes.


In [ ]:
rng = np.random.default_rng(42)
X = rng.random((1000, 50))
y = rng.integers(0, 10, size=1000)

# YOUR CODE — use tempfile.TemporaryDirectory() as tmp:
npy_size_kb  = None   # size of the .npy file in KB
npz_size_kb  = None   # size of the compressed .npz file in KB
X_loaded     = None   # reloaded X
y_loaded     = None   # reloaded y


In [ ]:
assert X_loaded is not None and y_loaded is not None
assert np.array_equal(X_loaded, X), "X round-trip failed"
assert np.array_equal(y_loaded, y), "y round-trip failed"
assert npy_size_kb > 0
assert npz_size_kb > 0
print(f"✓ Problem 08 passed")
print(f"  .npy size         : {npy_size_kb} KB")
print(f"  .npz compressed   : {npz_size_kb} KB")


---
### Problem 09 · Hard — rolling mean via stride tricks
Implement `rolling_mean(arr, W)` that computes the rolling mean over
window size `W` using stride tricks (zero-copy, no loops).

Then benchmark it against a naive Python loop implementation for
`arr` of length 100,000 with `W=50`.


In [ ]:
from numpy.lib.stride_tricks import as_strided

def rolling_mean_loop(arr, W):
    """Naive Python loop — reference implementation."""
    n = len(arr)
    out = np.empty(n - W + 1)
    for i in range(n - W + 1):
        out[i] = arr[i:i+W].mean()
    return out

# YOUR CODE
def rolling_mean_fast(arr, W):
    """Stride-trick implementation — no loops."""
    pass


In [ ]:
arr = np.random.default_rng(0).random(100_000)
W   = 50

expected = rolling_mean_loop(arr, W)
result   = rolling_mean_fast(arr, W)

assert result is not None
assert result.shape == expected.shape
assert np.allclose(result, expected)

t_loop, _ = timer(rolling_mean_loop, arr, W, repeat=3)
t_fast, _ = timer(rolling_mean_fast, arr, W, repeat=3)
speedup   = t_loop / t_fast

print(f"✓ Problem 09 passed")
print(f"  loop : {t_loop:.2f} ms")
print(f"  fast : {t_fast:.2f} ms")
print(f"  speedup: {speedup:.1f}x")


---
### Problem 10 · Hard — dtype memory audit
Given a list of arrays of various dtypes, write a function `memory_report`
that returns a summary dict for each array with keys:
`'shape'`, `'dtype'`, `'nbytes'`, `'mbytes'`.

Then find which array uses the most memory and what its dtype is.
Finally, show how much memory would be saved if all float64 arrays
were cast to float32.


In [ ]:
rng = np.random.default_rng(0)
arrays = {
    'weights'    : rng.normal(size=(512, 512)).astype(np.float64),
    'activations': rng.random((1000, 256)).astype(np.float32),
    'labels'     : rng.integers(0, 10, size=50_000).astype(np.int64),
    'mask'       : rng.random((1000, 1000)) > 0.5,  # bool
    'embeddings' : rng.normal(size=(10_000, 128)).astype(np.float64),
}

# YOUR CODE
def memory_report(arrays_dict):
    """Return a dict of dicts with shape, dtype, nbytes, mbytes per array."""
    pass

report        = None   # call memory_report(arrays)
largest_name  = None   # name of the array using the most memory
f64_saving_mb = None   # MB saved by casting all float64 to float32


In [ ]:
assert report is not None
for name, arr in arrays.items():
    assert name in report
    assert report[name]['shape']  == arr.shape
    assert report[name]['dtype']  == str(arr.dtype)
    assert report[name]['nbytes'] == arr.nbytes
    assert np.isclose(report[name]['mbytes'], arr.nbytes / 1e6)

assert largest_name in arrays
assert arrays[largest_name].nbytes == max(a.nbytes for a in arrays.values())

total_f64 = sum(a.nbytes for a in arrays.values() if a.dtype == np.float64)
assert np.isclose(f64_saving_mb, total_f64 / 2 / 1e6)

print(f"✓ Problem 10 passed")
print(f"  largest array  : {largest_name} ({report[largest_name]['mbytes']:.2f} MB)")
print(f"  float64→float32 saving: {f64_saving_mb:.2f} MB")
for name, info in report.items():
    print(f"  {name:<14} {str(info['dtype']):<10} {info['mbytes']:.2f} MB")
